# 13 Grouping and summarising

Split the data into groups, apply a calculation, combine the results.

In [1]:
import pandas as pd

patients = pd.read_csv("../data/patients.csv", parse_dates=["enrolled"])
patients.groupby("site")["age"].mean().round(1)

site
Gulu       49.3
Jinja      38.4
Kampala    38.3
Mbale      38.2
Mbarara    48.1
Name: age, dtype: float64

## Several statistics

`agg()` with named results gives tidy column names.

In [2]:
summary = (
    patients
    .groupby("site")
    .agg(
        n=("patient_id", "count"),
        mean_age=("age", "mean"),
        median_weight=("weight_kg", "median"),
        missing_weight=("weight_kg", lambda s: s.isna().sum()),
        pct_female=("sex", lambda s: (s == "F").mean() * 100),
    )
    .round(1)
    .sort_values("n", ascending=False)
)
summary

,n,mean_age,median_weight,missing_weight,pct_female
site,,,,,
Kampala,97,38.3,61.9,1,50.5
Mbarara,68,48.1,62.8,3,54.4
Gulu,51,49.3,62.2,3,58.8
Jinja,48,38.4,62.1,4,50.0
Mbale,36,38.2,59.4,1,36.1


## Grouping by more than one column

In [3]:
patients.groupby(["site", "sex"])["age"].agg(["count", "mean"]).round(1).head(6)

count  mean
site    sex             
Gulu    F       30  46.0
        M       21  54.0
Jinja   F       24  34.5
        M       24  42.3
Kampala F       49  34.0
        M       48  42.6

`reset_index()` turns the group labels back into ordinary columns, which is easier to plot or save:

In [4]:
patients.groupby(["site", "sex"], as_index=False)["age"].mean().head()

,site,sex,age
0,Gulu,F,45.966667
1,Gulu,M,54.000000
2,Jinja,F,34.500000
3,Jinja,M,42.291667
4,Kampala,F,34.040816


## Counts and cross tables

In [5]:
pd.crosstab(patients["site"], patients["hiv_status"], margins=True)

hiv_status,Negative,Positive,Unknown,All
site,,,,
Gulu,42,6,3,51
Jinja,37,5,6,48
Kampala,73,17,7,97
Mbale,27,6,3,36
Mbarara,52,10,6,68
All,231,44,25,300


In [6]:
pd.crosstab(patients["site"], patients["hiv_status"], normalize="index").round(3) * 100

hiv_status,Negative,Positive,Unknown
site,,,
Gulu,82.4,11.8,5.9
Jinja,77.1,10.4,12.5
Kampala,75.3,17.5,7.2
Mbale,75.0,16.7,8.3
Mbarara,76.5,14.7,8.8


## transform: a group value on every row

`agg` gives one row per group. `transform` gives a value for every original row,
for example each patient's age relative to their site's mean.

In [7]:
patients["site_mean_age"] = patients.groupby("site")["age"].transform("mean").round(1)
patients["age_vs_site"] = patients["age"] - patients["site_mean_age"]
patients[["patient_id", "site", "age", "site_mean_age", "age_vs_site"]].head()

,patient_id,site,age,site_mean_age,age_vs_site
0,P0001,Kampala,30,38.3,-8.3
1,P0002,Mbarara,81,48.1,32.9
2,P0003,Gulu,24,49.3,-25.3
3,P0004,Gulu,8,49.3,-41.3
4,P0005,Gulu,74,49.3,24.7


## Time-based groups

In [8]:
by_month = (
    patients
    .groupby(patients["enrolled"].dt.to_period("M"))
    .size()
    .rename("enrolled")
)
by_month.head()

enrolled
2025-01    25
2025-02    21
2025-03    31
2025-04    38
2025-05    19
Freq: M, Name: enrolled, dtype: int64

In [9]:
patients.set_index("enrolled").resample("QE")["patient_id"].count()

enrolled
2025-03-31    77
2025-06-30    78
2025-09-30    77
2025-12-31    68
Freq: QE-DEC, Name: patient_id, dtype: int64

## filter: keep whole groups

In [10]:
big_sites = patients.groupby("site").filter(lambda g: len(g) >= 50)
big_sites["site"].unique()

<StringArray>
['Kampala', 'Mbarara', 'Gulu']
Length: 3, dtype: str

## Ranking within groups

In [11]:
patients["age_rank_in_site"] = patients.groupby("site")["age"].rank(ascending=False, method="first")
patients.loc[patients["age_rank_in_site"] <= 2, ["site", "patient_id", "age"]].sort_values(["site", "age"])

,site,patient_id,age
71,Gulu,P0072,81
256,Gulu,P0257,83
196,Jinja,P0197,83
285,Jinja,P0286,83
66,Kampala,P0067,83
125,Kampala,P0126,84
159,Mbale,P0160,79
72,Mbale,P0073,81
88,Mbarara,P0089,83
14,Mbarara,P0015,84
